In [3]:
import duckdb
from pathlib import Path
import pandas as pd
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')



con = duckdb.connect()
data_dir = Path(r'C:\Users\Мартинсон Диана\metric tree\data\raw\ml-25m')
ratings_path = data_dir / 'ratings.csv'
movies_path = data_dir / 'movies.csv'

# Проверка наличия файлов
if not ratings_path.exists() or not movies_path.exists():
    raise FileNotFoundError(f"Ошибка: Файлы данных не найдены в папке {data_dir}")


con.execute(f"CREATE TABLE IF NOT EXISTS raw_ratings AS SELECT * FROM read_csv_auto('{ratings_path}')")
con.execute(f"CREATE TABLE IF NOT EXISTS raw_movies AS SELECT * FROM read_csv_auto('{movies_path}')")

con.execute("""
    CREATE OR REPLACE TABLE user_activity AS
    SELECT 
        r.userId, r.movieId, r.rating, r.timestamp,
        to_timestamp(r.timestamp) AS datetime,
        CAST(to_timestamp(r.timestamp) AS DATE) AS date,
        DATE_PART('year', to_timestamp(r.timestamp)) AS year,
        DATE_PART('month', to_timestamp(r.timestamp)) AS month,
        DATE_PART('hour', to_timestamp(r.timestamp)) AS hour,
        m.title, m.genres
    FROM raw_ratings r
    LEFT JOIN raw_movies m ON r.movieId = m.movieId
""")


# --- Общая статистика ---
total_ratings = con.execute("SELECT COUNT(*) FROM raw_ratings").fetchone()[0]
total_users = con.execute("SELECT COUNT(DISTINCT userId) FROM raw_ratings").fetchone()[0]
total_movies = con.execute("SELECT COUNT(DISTINCT movieId) FROM raw_ratings").fetchone()[0]
avg_rating = con.execute("SELECT ROUND(AVG(rating), 2) FROM raw_ratings").fetchone()[0]
median_rating = con.execute("SELECT ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY rating), 2) FROM raw_ratings").fetchone()[0]

date_range = con.execute("SELECT MIN(date), MAX(date) FROM user_activity").fetchone()

# --- Engagement (Вовлеченность) ---
dau_stats = con.execute("""
    SELECT AVG(daily_users), MAX(daily_users), MIN(daily_users), PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY daily_users)
    FROM (SELECT date, COUNT(DISTINCT userId) as daily_users FROM user_activity GROUP BY date)
""").fetchone()

user_activity_stats = con.execute("""
    SELECT AVG(user_ratings), MEDIAN(user_ratings), AVG(user_movies), MAX(user_ratings)
    FROM (SELECT userId, COUNT(*) as user_ratings, COUNT(DISTINCT movieId) as user_movies
          FROM raw_ratings GROUP BY userId)
""").fetchone()

# --- Quality (Качество) ---
rating_dist = con.execute("""
    SELECT rating, COUNT(*) as count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM raw_ratings GROUP BY rating ORDER BY rating
""").df()

top_movies = con.execute("""
    SELECT m.title, COUNT(*) as ratings_count, ROUND(AVG(r.rating), 2) as avg_rating
    FROM raw_ratings r
    JOIN raw_movies m ON r.movieId = m.movieId
    GROUP BY m.movieId, m.title
    HAVING COUNT(*) >= 100
    ORDER BY avg_rating DESC, ratings_count DESC
    LIMIT 10
""").df()

# --- Retention (Удержание) ---
retention_d1 = con.execute("""
    WITH user_first AS (SELECT userId, MIN(date) as first_day FROM user_activity GROUP BY userId),
    retention_check AS (
        SELECT ufd.userId, MAX(CASE WHEN ua.date = ufd.first_day + INTERVAL '1 day' THEN 1 ELSE 0 END) as ret 
        FROM user_first ufd LEFT JOIN user_activity ua ON ufd.userId = ua.userId GROUP BY ufd.userId
    )
    SELECT ROUND(SUM(ret) * 100.0 / COUNT(*), 1) FROM retention_check
""").fetchone()[0]

retention_d7 = con.execute("""
    WITH user_first AS (SELECT userId, MIN(date) as first_day FROM user_activity GROUP BY userId),
    retention_check AS (
        SELECT ufd.userId, MAX(CASE WHEN ua.date BETWEEN ufd.first_day AND ufd.first_day + INTERVAL '6 days' THEN 1 ELSE 0 END) as ret 
        FROM user_first ufd LEFT JOIN user_activity ua ON ufd.userId = ua.userId GROUP BY ufd.userId
    )
    SELECT ROUND(SUM(ret) * 100.0 / COUNT(*), 1) FROM retention_check
""").fetchone()[0]

churn_rate = 100 - retention_d1


print("\n FINAL REPORT")


print(f"\nОБЩАЯ СТАТИСТИКА:")
print(f"  Всего оценок: {total_ratings:,}")
print(f"  Пользователей: {total_users:,}")
print(f"  Фильмов: {total_movies:,}")
print(f"  Средний рейтинг: {avg_rating}")
print(f"  Период данных: {date_range[0]} - {date_range[1]}")

print(f"\nМЕТРИКИ ВОВЛЕЧЕННОСТИ:")
print(f"  Средний DAU: {dau_stats[0]:,.0f}")
print(f"  Оценок на пользователя: {user_activity_stats[0]:.1f}")

print(f"\nМЕТРИКИ УДЕРЖАНИЯ:")
print(f"  Day-1 Retention: {retention_d1}%")
print(f"  Day-7 Retention: {retention_d7}%")
print(f"  Churn Rate: {churn_rate:.1f}%")



reports_dir = Path('reports')
reports_dir.mkdir(exist_ok=True)

# Сохранение CSV
summary_data = [
    ['Общие', 'Всего оценок', f"{total_ratings:,}"],
    ['Общие', 'Пользователей', f"{total_users:,}"],
    ['Вовлеченность', 'Средний DAU', f"{dau_stats[0]:,.0f}"],
    ['Удержание', 'Day-1 Retention', f"{retention_d1}%"],
    ['Удержание', 'Day-7 Retention', f"{retention_d7}%"],
]
summary_df = pd.DataFrame(summary_data, columns=['Категория', 'Метрика', 'Значение'])
summary_df.to_csv(reports_dir / 'final_summary.csv', index=False, encoding='utf-8-sig')

# Генерация HTML
html_content = f"""
<!DOCTYPE html>
<html lang="ru">
<head>
    <meta charset="UTF-8">
    <title>Финальный отчет MovieLens</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 40px; line-height: 1.6; color: #333; }}
        h1 {{ color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px; }}
        h2 {{ color: #2980b9; margin-top: 30px; }}
        .stat-box {{ background: #f8f9fa; padding: 15px; margin: 10px 0; border-left: 5px solid #3498db; }}
        .stat-value {{ font-size: 24px; font-weight: bold; color: #2980b9; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 20px; }}
        th, td {{ padding: 12px; border: 1px solid #ddd; text-align: left; }}
        th {{ background-color: #3498db; color: white; }}
    </style>
</head>
<body>
    <h1>Финальный аналитический отчет: MovieLens</h1>
    <p>Дата генерации: {datetime.now().strftime('%d.%m.%Y %H:%M')}</p>
    
    <h2>1. Общие показатели</h2>
    <div class="stat-box">
        Всего оценок: <span class="stat-value">{total_ratings:,}</span>
    </div>
    <div class="stat-box">
        Активных пользователей: <span class="stat-value">{total_users:,}</span>
    </div>
    <div class="stat-box">
        Средний рейтинг: <span class="stat-value">{avg_rating}</span>
    </div>

    <h2>2. Вовлеченность и Удержание</h2>
    <div class="stat-box">
        Средний DAU: <span class="stat-value">{dau_stats[0]:,.0f}</span>
    </div>
    <div class="stat-box">
        Day-1 Retention: <span class="stat-value">{retention_d1}%</span>
    </div>
    <div class="stat-box">
        Churn Rate: <span class="stat-value">{churn_rate:.1f}%</span>
    </div>

    <h2>3. Топ фильмов (по рейтингу)</h2>
    <table>
        <tr><th>Фильм</th><th>Рейтинг</th><th>Кол-во оценок</th></tr>
        {''.join([f"<tr><td>{row['title']}</td><td>{row['avg_rating']}</td><td>{row['ratings_count']}</td></tr>" for _, row in top_movies.head(5).iterrows()])}
    </table>

    <h2>4. Детальная таблица метрик</h2>
    {summary_df.to_html(index=False)}

    <p style="margin-top: 50px; color: #7f8c8d; text-align: center;">
        Отчет сгенерирован автоматически.
    </p>
</body>
</html>
"""

with open(reports_dir / 'final_report.html', 'w', encoding='utf-8') as f:
    f.write(html_content)

print(f"\n Отчет сохранен в: {reports_dir}")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


 FINAL REPORT

ОБЩАЯ СТАТИСТИКА:
  Всего оценок: 25,000,095
  Пользователей: 162,541
  Фильмов: 59,047
  Средний рейтинг: 3.53
  Период данных: 1995-01-09 - 2019-11-21

МЕТРИКИ ВОВЛЕЧЕННОСТИ:
  Средний DAU: 162
  Оценок на пользователя: 153.8

МЕТРИКИ УДЕРЖАНИЯ:
  Day-1 Retention: 18.8%
  Day-7 Retention: 100.0%
  Churn Rate: 81.2%

 Отчет сохранен в: reports
